# Data coverage & quality inspection

Quick health check of the curated Massive store: what is there, gaps versus the NYSE calendar, and sanity checks on bars.

In [ ]:
import datetime as dt
import polars as pl
import matplotlib.pyplot as plt

from collective_alpha.storage.catalog import Catalog
from collective_alpha.calendar import trading_days

cat = Catalog()
cat.tables()

## Daily bars: rows and tickers per day

In [ ]:
daily = cat.sql('''
    select date, count(*) as rows, count(distinct ticker) as tickers,
           sum(volume) as volume
    from day_aggs group by date order by date
''').pl()
daily.tail()

In [ ]:
ax = daily.to_pandas().set_index('date')[['tickers']].plot(figsize=(12, 3), title='tickers per day (day_aggs)')

## Missing trading days

In [ ]:
have = set(daily['date'].to_list())
expected = trading_days(daily['date'].min(), daily['date'].max())
missing = [d for d in expected if d not in have]
print(len(missing), 'missing days'); missing[:20]

## Minute bars: bars per day and session boundaries

In [ ]:
minute = cat.sql('''
    select date, count(*) as rows, count(distinct ticker) as tickers,
           min(ts_ny) as first_bar, max(ts_ny) as last_bar
    from minute_aggs group by date order by date
''').pl()
minute.tail()

## Sanity checks on OHLC

In [ ]:
cat.sql('''
    select count(*) filter (where high < low) as high_lt_low,
           count(*) filter (where open > high or open < low) as open_outside,
           count(*) filter (where close > high or close < low) as close_outside,
           count(*) filter (where volume <= 0) as nonpositive_volume,
           count(*) as total
    from day_aggs
''').pl()

## Reference tables

In [ ]:
cat.sql('select active, count(*) as n from tickers group by active').pl()

In [ ]:
cat.sql('select type, count(*) as n from tickers where active group by type order by n desc limit 15').pl()

In [ ]:
cat.sql('select * from splits order by execution_date desc limit 10').pl()